# 面试题：如何从零实现 R-GCN，并正确处理知识图谱的逆关系？

## 面试回答主线

R-GCN 的关键不是“图上做一次平均”，而是每种关系拥有独立变换矩阵：目标节点按关系分别聚合邻居消息，再加 self-loop。若邻接定义为 `A[目标, 来源]=1`，入度归一化应逐目标行完成。创建逆关系时必须先对原始边转置，再对新的目标节点重新归一化；把已经归一化的正向矩阵直接转置，在多入度节点上会改变消息质量。

本实验让订单始终同时具有 `使用设备` 和 `购买商户` 两类业务事件，关系类型由事件字段决定，与未来拒付标签无关。模型只在最后读取 outcome 标签训练，不会再用 gold label 决定边类型。

## 真实案例：设备异常分与商户保障分共同预测订单拒付

每条订单事件包含设备历史异常分、商户售后保障分和金额档位。设备异常是风险证据，商户保障是保护证据；二者数值恰好成对，使忽略关系语义的模型只能看到相同均值。训练集含 12 个已结案订单，测试集含 6 个新强度订单。数值是脱敏教学构造，关系方向模拟事件图，不能外推为真实风控收益。

In [1]:
import torch  # 导入 PyTorch 以构造多关系图并执行真实反向传播。
from torch import nn  # 导入基础模块和可学习参数。
import torch.nn.functional as F  # 导入底层交叉熵和激活函数。
torch.set_num_threads(1)  # 固定小型图实验使用单线程。
torch.manual_seed(59)  # 固定参数初始化和训练输出。
train_records = [  # 定义十二个已结案训练订单且每对金额相同。
    ("T01", 0.90, 0.10, 0.60, 1),  # 设备异常显著高于商户保障且后续拒付。
    ("T02", 0.10, 0.90, 0.60, 0),  # 相同信号和但关系角色交换且后续正常。
    ("T03", 0.82, 0.18, 0.35, 1),  # 高设备风险订单。
    ("T04", 0.18, 0.82, 0.35, 0),  # 高商户保障订单。
    ("T05", 0.74, 0.26, 0.80, 1),  # 中高设备风险订单。
    ("T06", 0.26, 0.74, 0.80, 0),  # 中高商户保障订单。
    ("T07", 0.68, 0.32, 0.45, 1),  # 设备证据略占优势且拒付。
    ("T08", 0.32, 0.68, 0.45, 0),  # 商户证据略占优势且正常。
    ("T09", 0.88, 0.12, 0.70, 1),  # 另一高设备风险样本。
    ("T10", 0.12, 0.88, 0.70, 0),  # 对应高保障正常样本。
    ("T11", 0.62, 0.38, 0.25, 1),  # 较难的设备风险正样本。
    ("T12", 0.38, 0.62, 0.25, 0),  # 较难的商户保障负样本。
]  # 完成训练订单定义。
test_records = [  # 定义六个未参与参数更新的新信号强度订单。
    ("E01", 0.77, 0.23, 0.55, 1),  # 新强度设备风险正样本。
    ("E02", 0.23, 0.77, 0.55, 0),  # 同金额与信号和的关系交换负样本。
    ("E03", 0.71, 0.29, 0.40, 1),  # 中等设备风险正样本。
    ("E04", 0.29, 0.71, 0.40, 0),  # 对应商户保障负样本。
    ("E05", 0.58, 0.42, 0.75, 1),  # 接近边界的设备风险正样本。
    ("E06", 0.42, 0.58, 0.75, 0),  # 接近边界的商户保障负样本。
]  # 完成留出订单定义。
print("订单  设备异常  商户保障  金额档  未来outcome")  # 输出事件与标签表头。
for record in train_records[:4] + test_records:  # 展示部分训练事件和全部测试事件。
    print(f"{record[0]:<3}     {record[1]:.2f}      {record[2]:.2f}     {record[3]:.2f}      {'拒付' if record[4] else '正常'}")  # 展示关系两端证据和独立 outcome。
print("所有订单边类型固定为：订单→设备、订单→商户；不会按 outcome 改名。")  # 明确关系来源独立于标签。

订单  设备异常  商户保障  金额档  未来outcome
T01     0.90      0.10     0.60      拒付
T02     0.10      0.90     0.60      正常
T03     0.82      0.18     0.35      拒付
T04     0.18      0.82     0.35      正常
E01     0.77      0.23     0.55      拒付
E02     0.23      0.77     0.55      正常
E03     0.71      0.29     0.40      拒付
E04     0.29      0.71     0.40      正常
E05     0.58      0.42     0.75      拒付
E06     0.42      0.58     0.75      正常
所有订单边类型固定为：订单→设备、订单→商户；不会按 outcome 改名。


## 图构建：从业务事件生成原始邻接，再独立归一化逆关系

每个订单、设备证据和商户证据各占一个节点。节点特征为 `[信号值, 金额档, 是否订单]`；订单到设备、订单到商户两条正向边由事件字段产生。为了让证据回传订单，代码用 `raw.T` 创建逆边，然后对逆边重新做行归一化。

In [2]:
RELATION_NAMES = ["订单到设备", "设备到订单", "订单到商户", "商户到订单"]  # 定义四种正反业务关系。
def normalize_incoming(raw_adjacency):  # 按目标节点入度归一化原始邻接。
    row_degree = raw_adjacency.sum(dim=1, keepdim=True)  # 统计每个目标节点收到的当前关系边数。
    return raw_adjacency / row_degree.clamp_min(1.0)  # 对有边目标做均值聚合且保留零行。
def build_event_graph(records):  # 从订单字段构造与 outcome 无关的多关系图。
    node_count = len(records) * 3  # 为每条订单分配订单、设备和商户三个节点。
    features = torch.zeros(node_count, 3)  # 创建节点特征矩阵。
    labels = torch.tensor([record[4] for record in records], dtype=torch.long)  # 单独保存未来结案 outcome。
    case_indices = torch.arange(len(records), dtype=torch.long)  # 订单节点统一放在图前部。
    device_offset = len(records)  # 计算设备证据节点起始索引。
    merchant_offset = len(records) * 2  # 计算商户证据节点起始索引。
    raw_order_to_device = torch.zeros(node_count, node_count)  # 创建订单到设备原始邻接。
    raw_order_to_merchant = torch.zeros(node_count, node_count)  # 创建订单到商户原始邻接。
    for order_index, record in enumerate(records):  # 遍历订单事件并写入节点与边。
        device_index = device_offset + order_index  # 定位当前订单的设备证据节点。
        merchant_index = merchant_offset + order_index  # 定位当前订单的商户证据节点。
        features[order_index] = torch.tensor([0.0, record[3], 1.0])  # 写入订单金额与节点身份特征。
        features[device_index] = torch.tensor([record[1], 0.0, 0.0])  # 写入独立设备异常观测。
        features[merchant_index] = torch.tensor([record[2], 0.0, 0.0])  # 写入独立商户保障观测。
        raw_order_to_device[device_index, order_index] = 1.0  # 根据使用设备事件写入正向边。
        raw_order_to_merchant[merchant_index, order_index] = 1.0  # 根据购买商户事件写入正向边。
    raw_device_to_order = raw_order_to_device.transpose(0, 1).clone()  # 先转置原始设备事件得到逆关系。
    raw_merchant_to_order = raw_order_to_merchant.transpose(0, 1).clone()  # 先转置原始商户事件得到逆关系。
    raw_relations = [raw_order_to_device, raw_device_to_order, raw_order_to_merchant, raw_merchant_to_order]  # 组合四个原始关系邻接。
    normalized_relations = [normalize_incoming(raw) for raw in raw_relations]  # 对每个方向的新目标节点重新归一化。
    return features, normalized_relations, raw_relations, case_indices, labels  # 返回特征、正确信息流与独立标签。
train_features, train_relations, train_raw_relations, train_case_indices, train_labels = build_event_graph(train_records)  # 构造训练事件图。
test_features, test_relations, test_raw_relations, test_case_indices, test_labels = build_event_graph(test_records)  # 构造完全分离的测试事件图。
print("训练/测试节点特征形状：", tuple(train_features.shape), tuple(test_features.shape))  # 展示两张互不连通的图规模。
print("训练图四种关系边数：", [int(raw.sum()) for raw in train_raw_relations])  # 展示每个订单始终具有相同关系集合。
print("测试订单1收到的逆关系来源：设备节点", int(test_relations[1][0].argmax()), "商户节点", int(test_relations[3][0].argmax()))  # 展示证据通过独立逆边流入订单。

训练/测试节点特征形状： (36, 3) (18, 3)
训练图四种关系边数： [12, 12, 12, 12]
测试订单1收到的逆关系来源：设备节点 6 商户节点 12


## Baseline（基线）：忽略关系类型的单邻接 GCN

基线把“设备到订单”和“商户到订单”合并后统一平均。因为每对样本的设备分与商户分之和相同，它无法知道哪个值是风险证据、哪个值是保护证据。基线仍使用真实训练和同一测试准确率，而不是直接假定会失败。

In [3]:
class RelationAgnosticGCN(nn.Module):  # 定义忽略事件类型的单关系基线。
    def __init__(self, input_dim=3, hidden_dim=10):  # 初始化共享邻居投影与分类头。
        super().__init__()  # 注册基础模块状态。
        self.self_weight = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.24)  # 创建订单自身特征投影。
        self.neighbor_weight = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.24)  # 创建所有邻居共享投影。
        self.hidden_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建隐藏层偏置。
        self.classifier_weight = nn.Parameter(torch.randn(hidden_dim, 2) * 0.20)  # 创建正常与拒付分类头。
        self.classifier_bias = nn.Parameter(torch.zeros(2))  # 创建分类偏置。
    def forward(self, features, raw_relations, case_indices):  # 执行关系无关消息聚合。
        combined_inverse = raw_relations[1] + raw_relations[3]  # 合并设备与商户到订单的两类原始逆边。
        combined_inverse = normalize_incoming(combined_inverse)  # 对合并邻接统一均值归一化。
        neighbor_message = combined_inverse @ features @ self.neighbor_weight  # 用同一矩阵处理两种语义相反的证据。
        hidden = torch.tanh(features @ self.self_weight + neighbor_message + self.hidden_bias)  # 融合自身与关系无关邻居。
        logits = hidden[case_indices] @ self.classifier_weight + self.classifier_bias  # 只对订单节点输出类别分数。
        return logits, hidden[case_indices], neighbor_message[case_indices]  # 返回预测和可审计中间表示。
def train_classifier(model, epochs, learning_rate, relation_aware):  # 用同一训练标签优化基线或 R-GCN。
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # 创建参数优化器。
    history = []  # 保存 loss、准确率与梯度轨迹。
    for epoch in range(epochs):  # 重复全图监督学习。
        optimizer.zero_grad()  # 清除上一轮梯度。
        if relation_aware:  # 为 R-GCN 传入分关系归一化邻接。
            logits, hidden, details = model(train_features, train_relations, train_case_indices)  # 执行多关系前向。
        else:  # 为基线传入原始邻接并在内部合并。
            logits, hidden, details = model(train_features, train_raw_relations, train_case_indices)  # 执行关系无关前向。
        loss = F.cross_entropy(logits, train_labels)  # 使用独立 outcome 标签计算订单分类损失。
        loss.backward()  # 把真实分类误差反向传播到图参数。
        gradient_norm = float(next(model.parameters()).grad.norm().detach())  # 记录首个消息参数的梯度。
        optimizer.step()  # 更新当前模型参数。
        accuracy = float((logits.argmax(dim=-1) == train_labels).float().mean())  # 计算当前训练订单准确率。
        history.append((float(loss.detach()), accuracy, gradient_norm))  # 保存训练过程诊断。
    return history  # 返回完整优化轨迹。
torch.manual_seed(61)  # 固定关系无关基线初始化。
baseline_model = RelationAgnosticGCN()  # 创建单关系 GCN 基线。
baseline_history = train_classifier(baseline_model, 450, 0.025, relation_aware=False)  # 真实训练关系无关模型。
with torch.no_grad():  # 关闭基线测试梯度。
    baseline_logits, baseline_hidden, baseline_messages = baseline_model(test_features, test_raw_relations, test_case_indices)  # 在独立测试图上推理。
    baseline_probabilities = torch.softmax(baseline_logits, dim=-1)[:, 1]  # 读取拒付概率。
baseline_predictions = (baseline_probabilities >= 0.5).long()  # 使用预先固定的二分类阈值。
baseline_accuracy = float((baseline_predictions == test_labels).float().mean())  # 计算关系无关测试准确率。
print("阶段  loss    训练准确率  邻居参数梯度")  # 输出基线训练轨迹表头。
for epoch in (0, 99, 449):  # 选择三个关键训练阶段。
    row = baseline_history[epoch]  # 读取当前阶段记录。
    print(f"{epoch + 1:>3}  {row[0]:>6.4f}    {row[1]:>6.1%}      {row[2]:>7.5f}")  # 展示关系丢失后的优化上限。
print("成对测试订单的基线邻居消息最大差：", [round(float((baseline_messages[index] - baseline_messages[index + 1]).abs().max()), 7) for index in range(0, len(test_records), 2)])  # 证明交换关系角色后合并均值完全相同。
print(f"关系无关 Baseline 测试准确率={baseline_accuracy:.1%}")  # 输出同口径基线结果。

阶段  loss    训练准确率  邻居参数梯度
  1  0.7007     50.0%      0.05271
100  0.6931     50.0%      0.00005
450  0.6931     50.0%      0.00000
成对测试订单的基线邻居消息最大差： [0.0, 0.0, 0.0]
关系无关 Baseline 测试准确率=50.0%


## 核心实现：逐关系权重、逐目标归一化与订单分类

`ManualRGCN` 为四种正反关系各维护一块权重。第一层一次计算 `A_r @ X @ W_r` 并累加；订单表示中真正起作用的是“设备到订单”和“商户到订单”两条逆关系，但它们使用不同矩阵，因此相同数值和不再等价。代码还返回逐关系消息范数，便于检查模型究竟依赖哪类事件。

In [4]:
class ManualRGCN(nn.Module):  # 定义手写关系图卷积订单分类器。
    def __init__(self, input_dim=3, hidden_dim=12, relation_count=4):  # 初始化 self-loop、逐关系矩阵与分类头。
        super().__init__()  # 注册基础模块状态。
        self.self_weight = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.22)  # 创建节点自身投影矩阵。
        self.relation_weights = nn.ParameterList([nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.22) for _ in range(relation_count)])  # 为每个关系创建独立消息矩阵。
        self.hidden_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建关系聚合后的偏置。
        self.classifier_weight = nn.Parameter(torch.randn(hidden_dim, 2) * 0.18)  # 创建订单 outcome 分类头。
        self.classifier_bias = nn.Parameter(torch.zeros(2))  # 创建分类偏置。
    def forward(self, features, normalized_relations, case_indices):  # 执行逐关系消息聚合和订单预测。
        messages = []  # 收集每种关系对全部节点的消息。
        for adjacency, weight in zip(normalized_relations, self.relation_weights):  # 对齐关系邻接与专属参数。
            messages.append(adjacency @ features @ weight)  # 按目标入度聚合当前关系来源节点。
        total_message = torch.stack(messages, dim=0).sum(dim=0)  # 累加四种关系但保留各自变换语义。
        hidden = torch.tanh(features @ self.self_weight + total_message + self.hidden_bias)  # 融合 self-loop 与关系消息。
        logits = hidden[case_indices] @ self.classifier_weight + self.classifier_bias  # 输出订单正常与拒付 logits。
        case_messages = torch.stack([message[case_indices] for message in messages], dim=1)  # 整理订单逐关系消息为样本、关系、隐藏维。
        return logits, hidden[case_indices], case_messages  # 返回预测、订单表示和关系级证据。
torch.manual_seed(67)  # 固定 R-GCN 参数初始化。
rgcn = ManualRGCN()  # 创建四关系图卷积模型。
initial_rgcn_logits, initial_rgcn_hidden, initial_rgcn_messages = rgcn(train_features, train_relations, train_case_indices)  # 执行未训练多关系前向。
print("订单 logits / hidden / relation message 形状：", tuple(initial_rgcn_logits.shape), tuple(initial_rgcn_hidden.shape), tuple(initial_rgcn_messages.shape))  # 展示公式对应真实张量。
print("订单T01四类消息范数：", [round(float(value), 4) for value in initial_rgcn_messages[0].norm(dim=-1)])  # 展示正向关系对订单为零而逆向关系带来证据。
print("订单T01/T02初始隐藏最大差：", round(float((initial_rgcn_hidden[0] - initial_rgcn_hidden[1]).abs().max()), 4))  # 展示关系专属矩阵能区分角色交换。

订单 logits / hidden / relation message 形状： (12, 2) (12, 12) (12, 4, 12)
订单T01四类消息范数： [0.0, 0.6786, 0.0, 0.057]
订单T01/T02初始隐藏最大差： 0.3009


## 真实训练与逐订单结果

R-GCN 和基线使用同一 12 个训练 outcome、同一 6 个测试 outcome 和 0.5 阈值。测试图是单独构建的断开图，训练 loss 不读取任何测试节点。下面输出 loss、梯度、逐订单拒付概率以及设备/商户逆关系消息范数。

In [5]:
rgcn_history = train_classifier(rgcn, 450, 0.025, relation_aware=True)  # 在十二个已结案订单上真实训练 R-GCN。
with torch.no_grad():  # 关闭独立测试图的梯度记录。
    rgcn_logits, rgcn_hidden, rgcn_messages = rgcn(test_features, test_relations, test_case_indices)  # 对六个新订单执行多关系推理。
    rgcn_probabilities = torch.softmax(rgcn_logits, dim=-1)[:, 1]  # 读取拒付概率。
rgcn_predictions = (rgcn_probabilities >= 0.5).long()  # 使用固定阈值得到最终预测。
rgcn_accuracy = float((rgcn_predictions == test_labels).float().mean())  # 计算留出订单准确率。
print("阶段  loss    训练准确率  self参数梯度")  # 输出 R-GCN 优化轨迹表头。
for epoch in (0, 49, 199, 449):  # 选择四个关键训练阶段。
    row = rgcn_history[epoch]  # 读取当前阶段训练诊断。
    print(f"{epoch + 1:>3}  {row[0]:>6.4f}    {row[1]:>6.1%}      {row[2]:>7.5f}")  # 展示多关系模型真实收敛过程。
print("订单  设备/商户  gold  Baseline概率/预测  RGCN概率/预测  设备逆消息/商户逆消息")  # 输出逐订单对照表头。
for index, record in enumerate(test_records):  # 遍历六个独立测试订单。
    device_message_norm = float(rgcn_messages[index, 1].norm())  # 读取设备到订单关系消息强度。
    merchant_message_norm = float(rgcn_messages[index, 3].norm())  # 读取商户到订单关系消息强度。
    print(f"{record[0]:<3}  {record[1]:.2f}/{record[2]:.2f}    {record[4]}       {float(baseline_probabilities[index]):.3f}/{int(baseline_predictions[index])}           {float(rgcn_probabilities[index]):.3f}/{int(rgcn_predictions[index])}          {device_message_norm:.3f}/{merchant_message_norm:.3f}")  # 展示同信号和下关系语义产生的差异。
print(f"测试准确率：关系无关 Baseline={baseline_accuracy:.1%}，R-GCN={rgcn_accuracy:.1%}")  # 汇总同数据同指标结果。

阶段  loss    训练准确率  self参数梯度
  1  0.6866     50.0%      0.03060
 50  0.0060    100.0%      0.00078
200  0.0009    100.0%      0.00001
450  0.0002    100.0%      0.00000
订单  设备/商户  gold  Baseline概率/预测  RGCN概率/预测  设备逆消息/商户逆消息
E01  0.77/0.23    1       0.500/0           1.000/1          2.962/0.860
E02  0.23/0.77    0       0.500/0           0.000/0          0.885/2.878
E03  0.71/0.29    1       0.500/0           1.000/1          2.731/1.084
E04  0.29/0.71    0       0.500/0           0.000/0          1.116/2.654
E05  0.58/0.42    1       0.500/1           0.989/1          2.231/1.570
E06  0.42/0.58    0       0.500/1           0.012/0          1.616/2.168
测试准确率：关系无关 Baseline=50.0%，R-GCN=100.0%


## 结果解读

关系无关基线把每对订单的两个信号平均成相同表示，因此即使真实训练也只能在成对标签间折中。R-GCN 不是因为“拒付订单换了一种关系”，而是因为每条订单始终拥有相同的设备与商户事件，模型学会两种事件角色对同一数值的不同含义。这个受控配对实验验证 relation-specific message passing，不代表线上风控泛化；真实评估仍需时间切分、反欺诈延迟标签和稳定性监控。

## 失败案例：先归一化正向邻接再转置

假设三个订单共同使用一个设备。正向 `订单→设备` 的设备行入度为 3，归一化后每条边为 1/3；若直接转置，这三个订单在逆向各只收到 1/3 消息。正确逆关系应先转置原始边，每个订单此时只有一个设备来源，再按新目标入度归一化为 1。

In [6]:
shared_raw_forward = torch.zeros(4, 4)  # 创建三个订单与一个共享设备的原始邻接。
shared_raw_forward[3, 0] = 1.0  # 写入订单零到共享设备的正向边。
shared_raw_forward[3, 1] = 1.0  # 写入订单一到共享设备的正向边。
shared_raw_forward[3, 2] = 1.0  # 写入订单二到共享设备的正向边。
wrong_inverse = normalize_incoming(shared_raw_forward).transpose(0, 1)  # 复现先归一化再转置的错误流程。
correct_inverse_raw = shared_raw_forward.transpose(0, 1).clone()  # 先对未归一化原始边建立逆关系。
correct_inverse = normalize_incoming(correct_inverse_raw)  # 再按逆关系的新目标订单重新归一化。
source_features = torch.tensor([[0.0], [0.0], [0.0], [0.9]])  # 给共享设备设置可读风险信号。
wrong_messages = wrong_inverse @ source_features  # 计算错误逆关系传给三个订单的消息。
correct_messages = correct_inverse @ source_features  # 计算正确逆关系传给三个订单的消息。
print("订单  错误逆边行和  正确逆边行和  错误消息  正确消息")  # 输出多入度反例表头。
for order_index in range(3):  # 遍历三个共享设备订单。
    print(f"{order_index:>2}       {float(wrong_inverse[order_index].sum()):.3f}          {float(correct_inverse[order_index].sum()):.3f}        {float(wrong_messages[order_index]):.3f}     {float(correct_messages[order_index]):.3f}")  # 展示错误流程把消息缩小三倍。
print("共享设备正向入度：", int(shared_raw_forward[3].sum()), "；逆向每个订单正确入度：", [int(correct_inverse_raw[index].sum()) for index in range(3)])  # 解释归一化目标已随反向发生变化。

订单  错误逆边行和  正确逆边行和  错误消息  正确消息
 0       0.333          1.000        0.300     0.900
 1       0.333          1.000        0.300     0.900
 2       0.333          1.000        0.300     0.900
共享设备正向入度： 3 ；逆向每个订单正确入度： [1, 1, 1]


## 生产差距与追问

真实知识图谱还要处理共享实体、时间戳、重复边、边权、未知关系、basis decomposition、邻居采样和 inductive 新节点。关系必须来自可审计的业务事件 schema，不能根据预测标签命名；历史拒付等标签衍生特征要做严格时间截断。邻接方向与归一化合同应写进单元测试，尤其要覆盖一对多、多对一和重复事件，否则“转置后看起来 shape 正确”仍可能静默改变消息尺度。

## 最小回归测试

In [7]:
assert len(test_records) >= 5  # 保证关系推理案例包含足够逐订单样本。
assert all(int(raw.sum()) == len(train_records) for raw in train_raw_relations)  # 保护所有关系都来自每条订单固定事件而非标签分支。
assert baseline_accuracy <= 0.6  # 保护关系无关基线确实无法区分成对角色交换。
assert rgcn_accuracy > baseline_accuracy  # 保护逐关系参数在同一数据上带来有效改善。
assert torch.allclose(correct_inverse[:3].sum(dim=1), torch.ones(3))  # 保护正确逆关系按新目标重新归一化。
assert float((wrong_messages[:3] - correct_messages[:3]).abs().max()) > 0.1  # 保护多入度反例真实暴露错误消息尺度。
print("最小回归测试通过：独立事件关系、关系专属消息与逆边重新归一化均保持有效。")  # 输出集中测试结论。

最小回归测试通过：独立事件关系、关系专属消息与逆边重新归一化均保持有效。
